# Plant Disease Detection System

This notebook trains the model used by the Flask backend. It reads images from the local `Dataset` folder and saves the trained artifact to `backend/models/plant_disease_model.pkl`, which is the file loaded by the platform when a user uploads a leaf image.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Week 2":
    PROJECT_ROOT = PROJECT_ROOT.parent

BACKEND_DIR = PROJECT_ROOT / "backend"
DATASET_ROOT = PROJECT_ROOT / "Dataset" / "New Plant Diseases Dataset(Augmented)" / "New Plant Diseases Dataset(Augmented)"
TRAIN_DIR = DATASET_ROOT / "train"
VALID_DIR = DATASET_ROOT / "valid"
MODEL_PATH = BACKEND_DIR / "models" / "plant_disease_model.pkl"

sys.path.insert(0, str(BACKEND_DIR))

print("Project root:", PROJECT_ROOT)
print("Train dir:", TRAIN_DIR)
print("Valid dir:", VALID_DIR)
print("Model output:", MODEL_PATH)
print("Train exists:", TRAIN_DIR.exists())
print("Valid exists:", VALID_DIR.exists())

In [ ]:
from train_pkl_model import collect_image_paths

train_samples, class_names = collect_image_paths(TRAIN_DIR)
valid_samples, valid_class_names = collect_image_paths(VALID_DIR)

if class_names != valid_class_names:
    raise ValueError("Training and validation folders contain different class names.")

print(f"Classes: {len(class_names)}")
print(f"Training images: {len(train_samples)}")
print(f"Validation images: {len(valid_samples)}")
class_names[:5]

## Train the backend model

The backend uses a lightweight scikit-learn classifier over compact image features. This keeps inference simple for the Flask API and avoids requiring a large neural-network artifact to serve predictions.

In [ ]:
from train_pkl_model import evaluate_model, train_model
import pickle

BATCH_SIZE = 256

classifier = train_model(train_samples, class_names, batch_size=BATCH_SIZE)
validation_accuracy = evaluate_model(classifier, valid_samples, class_names, batch_size=BATCH_SIZE)

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
with MODEL_PATH.open("wb") as file:
    pickle.dump(
        {
            "type": "sklearn_sgd_image_classifier",
            "model": classifier,
            "class_names": class_names,
            "validation_accuracy": validation_accuracy,
        },
        file,
    )

print(f"Saved model: {MODEL_PATH}")
print(f"Validation accuracy: {validation_accuracy:.4f}")

## Verify upload prediction

This mirrors what the Flask `/predict` route does after a user uploads an image.

In [ ]:
from model_utils import PredictionService

sample_image = next((PROJECT_ROOT / "Dataset" / "test" / "test").glob("*.JPG"))
service = PredictionService(model_path=MODEL_PATH)

with sample_image.open("rb") as image_file:
    result = service.predict(image_file)

print("Sample image:", sample_image.name)
print("Model loaded:", service.model_loaded)
result

## Run the platform

Start the backend from the project root with `python backend/app.py`, then start the frontend from `frontend` with `pnpm install` if needed and `pnpm dev`. The Predict page uploads the selected image to the Flask `/predict` endpoint and displays the disease, confidence, treatment, and prevention guidance.